# Learning Rate V.S. Temporal Difference Learning Steps Analysis

This notebook analyzes the relationship between learning rate and temporal difference learning steps.

## Objective

- Understand how different learning rates affect the number of training steps required for agent convergence in the ARC environment.
- Identify optimal learning rate in combination with TD learning steps for stable training.

## Methodology

### Dataset

Using the standard ARC dataset V1, we randomly select 100 training examples.

### Hyperparameter Grid

We will keep the default hyperparameters constant while varying only:

- Learning rate (e.g., 1e-3, 3e-4, 1e-4, 3e-5)
- Temporal difference learning steps (e.g., 1, 3, 5, 10)
- Training episodes = 10000
- Epochs = 10
- Batch size = 32

### Metrics to Track

- Training loss curves
- Reward progression
- Convergence speed
- Training stability (variance in loss/reward)
- Success rate (percentage of episodes where agent solves the task)


# Sample path list


In [1]:
import os


def load_arc_sample(arc_agi_1_path, arc_agi_2_path):
    """Loads a sample of ARC files from the ARC AGI datasets."""
    with open("./arc_sample.txt", "r") as f:
        return f.read().split("\n")


def create_arc_sample(arc_agi_1_path, arc_agi_2_path, sample_size=50):
    """Creates a sample of ARC files from the ARC AGI datasets."""
    import random

    random.seed(42)
    arc_sample = [
        os.path.join(arc_agi_1_path, arc_file)
        for arc_file in random.sample(os.listdir(arc_agi_1_path), sample_size)
    ]
    arc_sample += [
        os.path.join(arc_agi_2_path, arc_file)
        for arc_file in random.sample(os.listdir(arc_agi_2_path), sample_size)
    ]
    with open("./arc_sample.txt", "w") as f:
        f.write("\n".join(arc_sample))
    return arc_sample


ARC_AGI_1_DATASET_PATH = os.path.join(
    os.environ.get("ARC_AGI_1_DATASET_PATH"), "training"
)
ARC_AGI_2_DATASET_PATH = os.path.join(
    os.environ.get("ARC_AGI_2_DATASET_PATH"), "training"
)
try:
    arc_sample = load_arc_sample(ARC_AGI_1_DATASET_PATH, ARC_AGI_2_DATASET_PATH)
except FileNotFoundError:
    arc_sample = create_arc_sample(ARC_AGI_1_DATASET_PATH, ARC_AGI_2_DATASET_PATH)

# Setup tarball with arc samples


In [2]:
import os
import time

from boto3.session import Session
from sagemaker.core.training.configs import OutputDataConfig

from aws_sagemaker import utils
from aws_sagemaker.launch import SagemakerExperiment


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/andrestello/Library/Application Support/sagemaker/config.yaml


In [3]:
session = Session()

aws_sagemaker_bucket = os.getenv("AWS_SAGEMAKER_BUCKET")
sagemaker_role_arn = os.getenv("AWS_SAGEMAKER_ROLE_ARN")
image_uri = "{}:latest".format(os.getenv("AWS_ECR_IMAGE_URI"))
experiment_location = "lr_experiment"

tar_s3_uri = utils.create_and_upload_arc_tar(
    session,
    "arc_sample.txt",
    aws_sagemaker_bucket,
    os.path.join(experiment_location, "arc_sample.tar.gz"),
    overwrite=False,
)
print("Dataset uploaded to: ", tar_s3_uri)

[01/06/26 19:49:51] WARNING  Object rlarcworld/lr_experiment/arc_sample.tar.gz already exists; skipping ]8;id=492857;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/utils.py\utils.py]8;;\:]8;id=791192;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/utils.py#94\94]8;;\
                             upload.                                                                               

Dataset uploaded to:  s3://rlarcworld/lr_experiment/arc_sample.tar.gz


In [ ]:
experiments = (
    # "lr_1e3_nstep_1",
    # "lr_1e3_nstep_5",
    # "lr_1e3_nstep_15",
    # "lr_1e3_nstep_30",
    # "lr_1e3_nstep_50",
    # "lr_3e4_nstep_1",
    # "lr_3e4_nstep_5",
    # "lr_3e4_nstep_15",
    # "lr_3e4_nstep_30",
    # "lr_3e4_nstep_50",
    # "lr_1e4_nstep_1",
    # "lr_1e4_nstep_5",
    # "lr_1e4_nstep_15",
    # "lr_1e4_nstep_30",
    # "lr_1e4_nstep_50",
    # "lr_3e5_nstep_1",
    "lr_3e5_nstep_5",
    # "lr_3e5_nstep_15",
    # "lr_3e5_nstep_30",
    # "lr_3e5_nstep_50",
)
user_exp_name = "4_epochs_3e5_nstep5_only"
for experiment_name in experiments:
    output_data_config = OutputDataConfig(
        s3_output_path=f"s3://{aws_sagemaker_bucket}/lr_experiment/{user_exp_name}/{experiment_name}/",
        compression_type="GZIP",
    )
    sagemaker_experiment = SagemakerExperiment(
        role_arn=sagemaker_role_arn,
        experiment=experiment_name,
        experiments_config="./lr_experiment/experiments.yaml",
        output_data_config=output_data_config,
        training_image=image_uri,
        epochs=4,
        seed=42,
        instance_type="ml.g4dn.12xlarge",
        enable_spot_instance=True,
        instance_count=1,
        max_run=12 * 60 * 60,
        max_wait=24 * 60 * 60,
    )
    sagemaker_experiment.train(tar_s3_uri, wait=False, logs=True)
    print(f"Started experiment {experiment_name} - continuing in 30s...")
    time.sleep(30)

                    INFO     SageMaker Session:                                                        ]8;id=230128;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=878527;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#92\92]8;;\

                    INFO       - role arn: arn:aws:iam::921868060019:role/SagemakerFullAccessExp       ]8;id=933258;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=41333;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#93\93]8;;\

[01/06/26 19:49:56] INFO       - bucket: sagemaker-us-east-1-921868060019                              ]8;id=606834;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=524070;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#94\94]8;;\

                    INFO       - region: us-east-1                                                     ]8;id=556438;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=443085;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#95\95]8;;\

                    INFO     Source code:                                                              ]8;id=349776;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=770865;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#98\98]8;;\
                             source_dir='/Users/andrestello/Personal/RlArcProject/aws_sagemaker'                   
                             requirements=None entry_script='train.py' command=None                                
                             ignore_patterns=['*.ipynb', '.git', '.github', '.venv', '**/__pycache__',             
                             'tests']                                                                              

...

[01/06/26 19:50:19] INFO     Creating training_job resource.                                     ]8;id=882507;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=489018;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=721287;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=78940;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#340\340]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=199361;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=487985;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

[01/06/26 19:50:20] WARNING  Not displaing the training container logs as 'wait' is set to     ]8;id=978854;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=872457;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#762\762]8;;\
                             False.                                                                                

Started experiment lr_1e3_nstep_5 - continuing in 30s...
